Over Sampling
=============

Throughout the workspace, we have created 2D grids of (y,x) coordinates and input them into light profiles to
compute their image.

This calculates how much of the light profile's emission is observed with every 2D pixel defined on the grid.

However, there is a problem. If we only input the (y,x) coordinates at the centre of every pixel, we are not
evaluating how the entire light profile is observed within that pixel. If the light profile has a very steep gradient
in intensity from one edge of the pixel to the other, only evaluating the intensity at the centre of the pixel will
not give an accurate estimate of the total amount of light that falls within that pixel.

Over-sampling addresses this problem. Instead of evaluating the light profile at the centre of every pixel, we
evaluate it using a sub-grid of coordinates within every pixel and take the average of the intensity values.
Provided the sub-grid is high enough resolution that it "over-samples" the light profile within the pixel enough, this
will give an accurate estimate of the total intensity within the pixel.

__Default Over-Sampling__

Examples throughout the workspace use a default over-sampling set up that should ensure accurate results for any
analysis you have done. This default over-sampling is as follows:

- When evaluating the image of a galaxy, an adaptive over sampling grid is used which uses sub grids of size 8 x 8
in the central regions of the image, 4x4 further out and 2x2 beyond that.

- The outer regions retain 2x2 over-sampling (they never drop to 1x1) because the lensed source's light appears far
from the image centre. Compact source profiles, such as the Gaussians of a Multi Gaussian Expansion (MGE), are not
evaluated accurately with a single evaluation per pixel, and the resulting aliasing produces small-scale structure
in the likelihood which degrades gradient-based non-linear searches.

This guide will explain why these choices were made for the default over-sampling behaviour.

__Contents__

- **Illustration:** To illustrate over sampling, lets first create a uniform grid which does not over sample the.
- **Numerics:** Lets quickly check how the sub-grid is defined and stored numerically.
- **Images:** We now use over-sampling to compute the image of a Sersic light profile, which has a steep.
- **Adaptive Over Sampling:** We have shown that over-sampling is important for accurate image evaluation.
- **Multiple Lens Galaxies:** The analysis may contain multiple lens galaxies, each of which must be over-sampled accurately.
- **Ray Tracing:** So far, we have evaluated the image of a light profile using over-sampling on an unlensed uniform.
- **Default Ray Tracing:** By assuming the lens galaxy is near (0.0", 0.0"), it was simple to set up an adaptive over sampling.
- **Dataset & Modeling:** Throughout this guide, grid objects have been used to compute the image of light and mass profiles.
- **Pixelization:** Source galaxies can be reconstructed using pixelizations, which discretize the source's light onto.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt

__Illustration__

To illustrate over sampling, lets first create a uniform grid which does not over sample the pixels, using 
the `over_sample_size` input.

The input below uses `over_sample_size=1`, therefore each pixel is split into a sub-grid of 
size  `over_sample_size x over_sample_size` = `1 x 1`. This means the light profile is evaluated once at the centre of each pixel, 
which is equivalent to not over-sampling the grid at all.  

In [ ]:
grid_sub_1 = al.Grid2D.uniform(
    shape_native=(40, 40),
    pixel_scales=0.1,
    over_sample_size=1,
)

We now plot the grid, over laying a uniform grid of pixels to illustrate the area of each pixel within which we
want light profile intensities to be computed.

In [ ]:

aplt.plot_grid(grid=grid_sub_1, title="Grid (No Over-Sampling)")

We now create and plot a uniform grid which does over-sample the pixels, by inputting `over_sample_size=2`.

In [ ]:
grid_sub_2 = al.Grid2D.uniform(
    shape_native=(40, 40),
    pixel_scales=0.1,
    over_sample_size=2,
)

If we print `grid_sub_2` and its shape, we will find it is actually identical to `grid_sub_1`, despite the change
in `over_sample_size`:

In [ ]:
print(grid_sub_1)
print(grid_sub_2)
print(grid_sub_1.shape)
print(grid_sub_2.shape)

This is because the over sampled version of the grid is stored in a separate attribute, called `over_sampled`,
which we print below.

We see that for `grid_sub_1` and `grid_sub_2` the `over_sampled` grids are different, with the over sampled grid for
`grid_sub_2` containing four times as many entries corresponding to each pixel being sub-gridded in a 2 x 2 shape.

In [ ]:
print(grid_sub_1.over_sampled)
print(grid_sub_2.over_sampled)
print(grid_sub_1.over_sampled.shape)
print(grid_sub_2.over_sampled.shape)

We now plot the over sampled grid over the image, showing that each pixel is now split into a 2x2 sub-grid of 
coordinates. 

These are used to compute the intensity of the light profile and therefore more accurately estimate the total 
intensity within each pixel if there is a significant gradient in intensity within the pixel.

In the code below, it is the input `plot_over_sampled_grid=True` which ensures we plot the over sampled grid.

In [ ]:

aplt.plot_grid(grid=grid_sub_2, title="Over-Sampled Grid")

__Numerics__

Lets quickly check how the sub-grid is defined and stored numerically.

The first four pixels of this sub-grid correspond to the first four sub-pixels in the first pixel of the grid. 

The top-left pixel image above shows how the sub-pixels are spaced within the pixel. 

In [ ]:
print("(y,x) pixel 0 of grid_sub_1:")
print(grid_sub_1.over_sampled[0])
print("(y,x) pixel 0 of grid_sub_2:")
print(grid_sub_2.over_sampled[0])

We now confirm that the first four sub-pixels of the over-sampled grid correspond are contained within the 
first pixel of the grid.

In [ ]:
print("(y,x) pixel 0 (of original grid):")
print(grid_sub_2[0])
print("(y,x) sub-pixel 0 (of pixel 0):")
print(grid_sub_2.over_sampled[0])
print("(y,x) sub-pixel 1 (of pixel 0):")
print(grid_sub_2.over_sampled[1])
print("(y,x) sub-pixel 2 (of pixel 0):")
print(grid_sub_2.over_sampled[2])
print("(y,x) sub-pixel 3 (of pixel 0):")
print(grid_sub_2.over_sampled[3])

Numerically, the over-sampled grid contains the sub-pixel coordinates of every pixel in the grid, going from the 
first top-left pixel right and downwards to the bottom-right pixel. 

So the pixel to the right of the first pixel is the next 4 sub-pixels in the over-sampled grid, and so on.

__Images__

We now use over-sampling to compute the image of a Sersic light profile, which has a steep intensity gradient
at its centre which a lack of over-sampling does not accurately capture.

We create the light profile, input the two grids (with `over_sample_size=1` and `over_sample_size=2`) and compute the image of the
light profile using each grid. We then plot the residuals between the two images in order to show the difference
between the two images and thus why over-sampling is important.

Over sampling occurs automatically when a grid is input into a function like `image_2d_from`, therefore internally 
the line of code, `image_sub_2 = light.image_2d_from(grid=grid_sub_2)`, is evaluating the light profile using the
2 x 2 oversampled grid and internally binning it up in to fully perform over sampling.

In [ ]:
light = al.lp.Sersic(
    centre=(0.0, 0.0),
    ell_comps=(0.0, 0.0),
    intensity=1.0,
    effective_radius=0.2,
    sersic_index=3.0,
)

image_sub_1 = light.image_2d_from(grid=grid_sub_1)
image_sub_2 = light.image_2d_from(grid=grid_sub_2)

aplt.plot_array(array=image_sub_1, title="Image of Sersic Profile")

residual_map = image_sub_2 - image_sub_1

aplt.plot_array(array=residual_map, title="Over-Sampling Residuals")


In the central 4 pixels of the image, the residuals are large due to the steep intensity gradient of the Sersic
profile at its centre. 

The gradient in these pixels is so steep that evaluating the intensity at the centre of the pixel, without over 
sampling, does not accurately capture the total intensity within the pixel.

At the edges of the image, the residuals are very small, as the intensity gradient of the Sersic profile is very 
shallow and it is an accurate approximation to evaluate the intensity at the centre of the pixel.

The absolute value of the central residuals are 0.74, however it is difficult to assess whether this is a large or
small value. We can quantify this by dividing by the evaluated value of the Sersic image in each pixel in order
to compute the fractional residuals.

In [ ]:
fractional_residual_map = residual_map / image_sub_2


aplt.plot_array(
    array=fractional_residual_map, title="Fractional Over-Sampling Residuals"
)

The fractional residuals in the centre exceed 0.1, or 10%, which is a significant error in the image and
demonstrates why over-sampling is important.

Lets confirm sub-griding can converge to central residuals that are very small.

The fractional residuals with high levels of over-sampling are below 0.01, or 1%, which is sufficiently accurate
for most scientific purposes (albeit you should think carefully about the level of over-sampling you need for
your specific science case).

In [ ]:
grid_sub_16 = al.Grid2D.uniform(
    shape_native=(40, 40), pixel_scales=0.1, over_sample_size=16
)
grid_sub_32 = al.Grid2D.uniform(
    shape_native=(40, 40), pixel_scales=0.1, over_sample_size=32
)

image_sub_16 = light.image_2d_from(grid=grid_sub_16)
image_sub_32 = light.image_2d_from(grid=grid_sub_32)

residual_map = image_sub_32 - image_sub_16

aplt.plot_array(array=residual_map, title="Over-Sampling Reduces Residuals")

fractional_residual_map = residual_map / image_sub_32

aplt.plot_array(
    array=fractional_residual_map, title="Fractional Residuals With Over-Sampling"
)

__Adaptive Over Sampling__

We have shown that over-sampling is important for accurate image evaluation. However, there is a major drawback to
over-sampling, which is that it is computationally expensive. 

For example, for the 32x32 over-sampled grid above, 1024 sub-pixels are used in every pixel, which must all be 
evaluated using the Sersic light profile. The calculation of the image is therefore at least 1000 times slower than if
we had not used over-sampling.

Speeding up the calculation is crucial for model-fitting where the image is evaluated many times to fit the
model to the data.

Fortunately, there is a solution to this problem. We saw above that the residuals rapidly decrease away
from the centre of the light profile. Therefore, we only need to over-sample the central regions of the image,
where the intensity gradient is steep. We can use lower levels of over-sampling away from the centre, which
will be fast to evaluate.

Up to now, the `over_sample_size` input has been an integer, however it can also be an `ndarray` of values corresponding
to each pixel. We create an `ndarray` of values which are high in the centre, but reduce to 2 at the outskirts,
therefore providing high levels of over sampling where we need it whilst using lower values which are computationally
fast to evaluate at the outskirts.

Specifically, we define a 24 x 24 sub-grid within the central 0.3" of pixels, uses a 8 x 8 grid between
0.3" and 0.6" and a 2 x 2 grid beyond that. 

In [ ]:
over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=grid_sub_1, sub_size_list=[24, 8, 2], radial_list=[0.3, 0.6]
)

grid_adaptive = al.Grid2D.no_mask(
    values=grid_sub_1.native,
    pixel_scales=grid_sub_1.pixel_scales,
    over_sample_size=over_sample_size,
)


aplt.plot_grid(grid=grid_adaptive, title="Over-Sampled Grid")

print(over_sample_size)


Modeling uses masked grids, therefore the code below shows how we would create this adaptive over sample grid via 
a circular mask, which can be used for modeling.

Throughout the modeling examples in the workspace, we use this adaptive grid to ensure that the image of the
galaxy is evaluated accurately and efficiently.

In [ ]:
mask = al.Mask2D.circular(shape_native=(40, 40), pixel_scales=0.1, radius=5.0)

grid = al.Grid2D.from_mask(mask=mask)

over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=grid, sub_size_list=[24, 8, 2], radial_list=[0.3, 0.6]
)

grid_adaptive = al.Grid2D(values=grid, mask=mask, over_sample_size=over_sample_size)


aplt.plot_grid(grid=grid, title="Over-Sampled Grid")

We can compare this adaptive grid to the grid with over sampling of 32 x 32 to confine it produces low amounts
of residuals.

In [ ]:
grid_masked_sub_32 = al.Grid2D(values=grid, mask=mask, over_sample_size=32)

image_adaptive = light.image_2d_from(grid=grid_adaptive)
image_masked_sub_32 = light.image_2d_from(grid=grid_masked_sub_32)

residual_map = image_adaptive - image_masked_sub_32

fractional_residual_map = residual_map / image_masked_sub_32


aplt.plot_array(array=fractional_residual_map, title="Adaptive Fractional Residuals")

__Default Over-Sampling__

The default over-sampling scheme used by the source code is 4 x 4 uniform over sampling over the whole image. 

A uniform scheme is used, instead of the adaptive scheme above, because the adaptive scheme requires input knowledge of 
where the centre of the galaxy is (e.g. above the centre is at (0.0", 0.0").

Uniform over sampling is precise enough for many calculations, especially when you are simply performing quick 
calculations to investigate a problem. However, for detailed calculations you must ensure that high enough
levels of over sampling are used.

For modeling, all example scripts begin by switching to an adaptive over sampling scheme, as modeling assumes
the centre of the lens galaxy is at (0.0", 0.0").

__Multiple Lens Galaxies__

The analysis may contain multiple lens galaxies, each of which must be over-sampled accurately. 


There are two approaches you can take to over sampling multi-galaxy systems:

1) Use a high level of uniform over sampling over the full image.

2) Use an adaptive over sampling scheme with multiple centres of high over sampling levels, with the API shown below
  for two galaxies with centres (1.0, 0.0) and (-1.0, 0.0).

In [ ]:
over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=grid_sub_1,
    sub_size_list=[24, 8, 2],
    radial_list=[0.3, 0.6],
    centre_list=[(1.0, 0.0), (-1.0, 0.0)],
)

__Ray Tracing__

So far, we have evaluated the image of a light profile using over-sampling on an unlensed uniform grid. 

For lensing calculations, the grid is ray-traced via a mass model to an irregular grid in the source plane.

The over sampling of lensed images is therefore describe as follows: 

1) Splits each image-pixel into a sub-grid of pixels in the image-plane.
2) Ray trace this sub-grid of pixels using the mass model.
3) Evaluate the source light of each sub-pixel in the source-plane.
4) Bin up the values to evaluate the over sampled values.

For lensing calculations, over sampling therefore requires us to ray-trace (and therefore compute the deflecitons angles
of) many more (y,x) coordinates!

We now illustrate using over-sampling with a mass profile, noting that for lensing:

1) The fractional residuals due to differing over-sampling levels now occur in the lensed source's brightest multiply 
   imaged pixels. 
 
2) It is the combination of a rapidly changing source light profile and the magnification pattern of the mass model
   which requires over sampling. The mass model focuses many image-pixels to the source's brightest regions.

In [ ]:
mass = al.mp.Isothermal(centre=(0.0, 0.0), ell_comps=(0.1, 0.0), einstein_radius=1.0)

light = al.lp.Sersic(
    centre=(0.0, 0.0),
    ell_comps=(0.0, 0.0),
    intensity=1.0,
    effective_radius=0.2,
    sersic_index=3.0,
)

lens = al.Galaxy(redshift=0.5, mass=mass)

source = al.Galaxy(redshift=1.0, bulge=light)

tracer = al.Tracer(galaxies=[lens, source])

image_sub_1 = tracer.image_2d_from(grid=grid_sub_1)
image_sub_2 = tracer.image_2d_from(grid=grid_sub_2)

aplt.plot_array(array=image_sub_1, title="Source Image 1x1")

residual_map = image_sub_2 - image_sub_1

fractional_residual_map = residual_map / image_sub_2

aplt.plot_array(array=fractional_residual_map, title="Fractional Residuals")

__Default Ray Tracing__

By assuming the lens galaxy is near (0.0", 0.0"), it was simple to set up an adaptive over sampling grid which is
applicable to all strong lens dataset.

An adaptive oversampling grid cannot be defined for the lensed source because its light appears in different regions of
the image plane for each dataset. For this reason, the adaptive schemes used throughout the workspace never drop
below 2x2 over-sampling in their outer regions, so the lensed source is always evaluated with at least a 2x2 sub-grid
wherever its arcs fall. This matters most for compact source profiles, such as the Gaussians of a Multi Gaussian
Expansion (MGE): evaluating them once per pixel produces aliasing which puts small-scale structure on the likelihood
surface, degrading the performance of gradient-based non-linear searches.

__Adaptive Over Sampling__

There is a way to set up an adaptive over sampling grid for a lensed source, however it requries one to use and
understanding the advanced lens modeling feature search chaining.

An example of how to use search chaining to over sample sources efficient is provided in 
the `autolens_workspace/*/guides/advanced/over_sampling_chaining.ipynb` example.

__Dataset & Modeling__

Throughout this guide, grid objects have been used to compute the image of light and mass profiles and illustrate
over sampling.

If you are performing calculations with imaging data or want to fit a model to the data with a specific
over-sampling level, the `apply_over_sampling` method is used to update the over sampling scheme of the dataset.

The grid this is applied to is called `lp`, to indicate that it is the grid used to evaluate the emission of light
profiles for which this over sampling scheme is applied.

In [ ]:
dataset_name = "simple__no_lens_light"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/imaging/features/no_lens_light/simulator.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

# This can be any of the over-sampling objects we have used above.

dataset = dataset.apply_over_sampling(over_sample_size_lp=4)

__Pixelization__

Source galaxies can be reconstructed using pixelizations, which discretize the source's light onto a mesh,
for example a Voronoi mesh.

Over sampling is used by pixelizations in an analogous way to light profiles. By default, a 4 x 4 sub-grid is used,
whereby every image pixel is ray-traced on its 4 x 4 sub grid to the source mesh and fractional mappings are computed.

A different grid and over sampling scheme is applied to light profiles and pixelizations, which is why
there are separate inputs called `lp` and `pix`.

This is explained in more detail in the pixelization examples.

Here is an example of how to change the over sampling applied to a pixelization for a lens model fit:

In [ ]:
dataset = dataset.apply_over_sampling(over_sample_size_pixelization=4)

__Adaptive Pixelization Over-Sampling__

A uniform sub-size over-samples the whole mask equally, which is wasteful: the pixelization only gains accuracy
where the lensed source is bright. Once a pixelized fit has been performed, its source signal-to-noise map can
steer the over-sampling, using a sub-size of 4 where the source is detected and 2 everywhere else.

`galaxy_name_image_dict_via_result_from` returns, for each galaxy, its model image divided by the noise map --
that is, a signal-to-noise map. It is therefore thresholded directly.

```python
signal_to_noise_threshold = 3.0

source_image_raw = al.galaxy_name_image_dict_via_result_from(
    result=result
)["('galaxies', 'source')"]

over_sample_size_pixelization = al.Array2D(
    values=np.where(source_image_raw > signal_to_noise_threshold, 4, 2),
    mask=dataset.mask,
)

dataset = dataset.apply_over_sampling(
    over_sample_size_pixelization=over_sample_size_pixelization
)
```

The snippet is not run here because this guide has no pixelized fit to draw a `result` from. This is the SLaM
default: from SOURCE PIX PIPELINE 2 onwards the pixelization over-sampling is set this way, using the source
signal-to-noise map of SOURCE PIX PIPELINE 1. SOURCE PIX PIPELINE 1 itself keeps the dataset's uniform
sub-size. See `guides/modeling/slam_start_here.py` for the full pipeline.

__Oversampled PSF Convolution__

Everything above concerns over sampling the *evaluation* of light profiles — computing the image on a sub-grid
and binning it to pixel values before any instrument effect is applied. PSF convolution then blurs those
pixel values at the resolution of the image.

Convolution itself can also be over sampled. This matters when the PSF is undersampled by the detector (its
width is comparable to the pixel scale, as for HST or Euclid VIS imaging): blurring the binned image with a
pixel-scale PSF loses the sub-pixel structure of both the image and the PSF, and the two operations (bin then
convolve, versus convolve finely then bin) do not commute.

Supplying the PSF at a multiple of the image resolution and setting `convolve_over_sample_size` performs the
convolution on the over-sampled grid and bins the result back to image resolution:

In [ ]:
psf_fine = al.Convolver.from_gaussian(
    shape_native=(21, 21),  # twice the pixels of an 11x11 image-resolution kernel...
    pixel_scales=0.1 / 2,  # ...at half the pixel scale, so the same physical extent.
    sigma=0.05,
    normalize=True,
    convolve_over_sample_size=2,
)

dataset_fine = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

For a dataset, the sizes are set per operation, mirroring the `lp` / `pixelization` split above. The matching
evaluation `over_sample_size` must be a uniform integer equal to the convolution size — the values convolved at
the fine resolution must first be evaluated at exactly that resolution, so adaptive over sampling cannot be
combined with oversampled convolution (the code raises a clear error rather than silently degrading):

```python
dataset = al.Imaging(
    data=data,
    noise_map=noise_map,
    psf=psf_fine,
    over_sample_size_lp=2,
    over_sample_size_pixelization=2,
    convolve_over_sample_size_lp=2,
    convolve_over_sample_size_pixelization=2,
)
```

Fits then work across every model surface: standard light profiles, linear light profiles, operated light
profiles (which are added at image resolution unblurred, by definition) and pixelized source reconstructions.
Simulation is supported too — see the `__Oversampled PSF (Advanced)__` section of `scripts/imaging/simulator.py`.

__Adaptive Evaluation: the k x s Coupling__

Adaptive over sampling composes with oversampled convolution: every `over_sample_size` entry must be
*divisible* by `convolve_over_sample_size` (not equal to it). Each pixel is evaluated on its own adaptive
sub-grid of size k * s, the values are partially binned to a uniform image at the convolution resolution s,
convolved, and binned to image resolution — adaptive sampling does the integration accuracy, the uniform
intermediate does the convolution. The adaptive radial schemes above therefore work unchanged with an
oversampled PSF, provided their sizes are multiples of s (e.g. [32, 8, 2] with s=2). A non-divisible
combination raises a clear error. This holds for pixelized sources too, where the coupling is exact by
linearity of the mapping matrix.

__Limitations__

The following are not supported with an oversampled PSF and raise a clear error if combined with it:

 - Over sampling sizes not divisible by `convolve_over_sample_size` (e.g. 3 with s=2).
 - The sparse linear-algebra formalism (`apply_sparse_operator`), whose PSF products are precomputed at image
   resolution.
 - The fixed-linear-function preload (`data_linear_func_matrix`) used to accelerate some fixed-MGE fits.

The padded / unmasked visualization images (`padded_image_2d_from`, `unmasked_blurred_image_2d_from`) remain at
image resolution — visualization is unaffected by the convolution accuracy improvement.

The numerical verification of all of the above lives in
`autolens_workspace_test/scripts/imaging/convolution_over_sampled.py`, which pins the implementation to
brute-force reference calculations.